In [3]:
#MATH0001. Salary-Prediction-Using-Polynomial-Regression（SPUPR)
#Mathematical Comparison: Linear vs Fourth-Order Polynomial Regression for Salary Forecasting

# 1. import necessary pacakges
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.preprocessing import PolynomialFeatures
from ipywidgets import interact, IntSlider# for sliding bar


In [8]:
# 2. read the dataset
dataset_raw = pd.read_csv('Position_Salaries.csv')
dataset = dataset_raw[['Level','Salary']]

X = dataset[['Level']]
Y = dataset['Salary']

In [16]:
 # 3. Main Sliding bar function, allows you to adjust the degree of the polynomial for data fitting
def plot_polynomial_regression(degree:int):
  # 3.3.1 polynomial basis extension
  poly_features = PolynomialFeatures(degree = degree, include_bias=True)# include_bias=True gives Intercept 𝛽₀ immediately
  X_poly =poly_features.fit_transform(X)

  #3.3.4 build unified OLS Matrix estimation
  model = sm.OLS(Y,X_poly).fit()
  X_grid = np.arange(min(dataset['Level']),max(dataset['Level'])+0.1,0.1).reshape(-1,1)
  X_grid_poly = poly_features.transform(X_grid)
  y_grid_pred = model.predict(X_grid_poly)

  # 3.4 out-of-sample prediction for x = 6.5
  test_level = np.array([[6.5]])
  test_level_poly = poly_features.transform(test_level)
  pred_65 = model.predict(test_level_poly)[0]

  # 3.5 visualization
  plt.figure(figsize = (11,6))
  # plot the original points
  sns.scatterplot(x='Level',y='Salary',data = dataset,color = 'red',s=100,zorder=4, label='Observed Data (y_i)')

  # draw polynomial fitting curve
  plt.plot(X_grid, y_grid_pred, color = 'blue', linewidth = 2.5, zorder = 3,label = f'Polynomial Degree {degree} Fit')

  # label X=6.5 (the green X)
  plt.scatter(6.5,pred_65,color = 'green',marker = 'X',s=250,zorder = 5,label=f'Prediction for Level 6.5: ${pred_65:,.2f}')


  plt.title(f"Interactive Polynomial Regression Model (Degree = {degree})", fontsize=14, fontweight='bold')
  plt.xlabel("Job Level (x)", fontsize=12)
  plt.ylabel("Employee Salary (y)", fontsize=12)
  plt.xlim(0.5, 10.5)
  plt.ylim(-100000, 1100000)
  plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x)))) # 補上千分位逗號
  plt.grid(True, linestyle='--', alpha=0.5)
  plt.legend(loc='upper left', fontsize=11)
  plt.show()

  # Coefficient Table
  print("=" * 70)
  print(f"  REGRESSION MODEL FIT PERFORMANCE REPORT (Polynomial Degree d = {degree})")
  print("=" * 70)
  print(f"  Multiple R-squared  (R²) : {model.rsquared:.4f}")
  print(f"  Adjusted R-squared       : {model.rsquared_adj:.4f}")
  print(f"  Out-of-Sample Prediction :")
  print(f"    - Estimated Salary for Level 6.5: ${pred_65:,.2f}")
  print("=" * 70)
  print("  OLS INDEPENDENT VARIABLE COEFFICIENT MATRIX:")
  print("=" * 70)
  print(model.summary().tables[1]) # Prints coefficients, std err, t-stat, and p-values

In [19]:
# build the interactive sliding bar
interact(
    plot_polynomial_regression,
    degree = IntSlider(
        min=1,
        max=8,
        step=1,
        value=4, # default value
        description='Polynomial Degree (d):',
        continuous_update=False
    )
);

interactive(children=(IntSlider(value=4, continuous_update=False, description='Polynomial Degree (d):', max=8,…